# CP201A Lab 3: ACS Data in Python

**Fall 2026**

Today your Census API key goes to work. By the end of lab you will have pulled the
same ACS table at three different geographic scales, straight from the Census Bureau
into Python.

## Learning objectives

**Everyone**
* Save your Census API key once, so every notebook this semester can find it
* Read the anatomy of a Census API call: table, variables, geography, year
* Understand how census geography codes work, and where to look up your own
* Pull an ACS table at the census tract, city, and county scale
* Rename the Census Bureau's variable codes into labels you can actually read
* Check that what came back is what you asked for

**If you want more**
* Pull a whole state's worth of places in one call
* Look up variables and tables on your own

Next week (Lab 4) we take these estimates and deal with their margins of error.

## 0. Before we begin

This notebook uses the `census` package. Run the installation cell first. It makes sure
the package is available in your current Datahub environment. Then run the import cell
below it.

In [ ]:
%pip install -q census

In [ ]:
from census import Census
import pandas as pd
import numpy as np
import os

## 0.1 Saving your Census API key

You only need to do this once all semester. This cell saves your key to a small text
file in your home directory. Every notebook we use from here on will read your key from
that file automatically, so you never have to paste it again.

Before you run this cell, make sure you signed up for a key at
https://api.census.gov/data/key_signup.html and clicked the activation link in the
confirmation email from the Census Bureau.

**To run it:** click the cell, press Shift+Enter, and a prompt will appear below asking
for your key. Paste it in and press Enter. Nothing will appear as you paste, because the
input is hidden on purpose. That is normal.

Pasted the wrong thing? Just run the cell again. It overwrites the old file.

**You will not have to do this again after a kernel restart.** The key lives in a file, not
in memory, so restarting the kernel, logging out of Datahub, or pulling a fresh copy of the
notebook will not lose it. After a restart, run the imports and the cell below, and skip
this one.

**Why we do it this way:** your key is yours. Keeping it out of the notebook means you
can share, submit, or post a notebook without handing your key to anyone. Later in your
career you will handle keys that really matter, and this is the habit that protects them.

**If your key has not arrived yet:** sit with someone who has one and work through the
lab together, then come back and run it yourself once your key shows up. Everything in
here will still be waiting for you. Do get this sorted before Lab 4 on September 23.

In [ ]:
from getpass import getpass

key = getpass('Paste your Census API key and press Enter: ')

with open(os.path.expanduser('~/census_key.txt'), 'w') as f:
    f.write(key.strip())

print('Key saved. You will not need to do this again.')

Now let's confirm it worked. This next cell is the one that appears at the top of every
later notebook in this course. It reads your key from the file and hands it to the
`Census` object we will use to make requests.

In [ ]:
# This is the same code every later notebook will use to load your key.
with open(os.path.expanduser('~/census_key.txt')) as f:
    api_key = f.read().strip()

print('Key loaded. It starts with:', api_key[:4] + '...')

c = Census(key=api_key)

## 0.2 Two Python tools we will use today

Before the census data, two small things that will show up all afternoon.

### f-strings

An f-string lets you drop the value of a variable into the middle of a piece of text.
Put an `f` before the opening quote, then wrap the variable in curly braces.

In [ ]:
neighborhood = 'North Oakland'
print(f'Today we are looking at {neighborhood}.')

n_tracts = 5
print(f'{neighborhood} is made up of {n_tracts} census tracts.')

In [ ]:
# EXERCISE #1: try the line above without the f before the quote. What changes?

### for loops

A `for` loop repeats a block of code once for each item in a list. This is how we will
avoid copying and pasting the same line ten times with different variable names.

Read `for item in my_list:` as "for each item in my list, do the following."

In [ ]:
groups = ['White', 'Black', 'Asian', 'Hispanic or Latino']

for g in groups:
    print(f'We will need a column for {g}.')

In [ ]:
# EXERCISE #2: write a for loop that prints the numbers 1 through 5, each on its own line.
# Hint: range(1, 6) gives you those numbers.

## 1. The anatomy of a Census API call

Every request to the Census API answers four questions:

| Question | In our code | Today's answer |
| --- | --- | --- |
| Which survey? | `c.acs5` | ACS 5-year estimates |
| Which variables? | a list of variable codes | Table B03002 |
| Which geography? | the `for` and `in` arguments | census tracts, in Alameda County, in California |
| Which year? | `year=` | 2024, meaning the 2020 to 2024 5-year estimates |

Two things worth knowing about that last row. The ACS 5-year estimates are labeled by
their **final** year, so `year=2024` gets you data pooled from 2020 through 2024. And
census geographies are **nested**: a tract sits inside a county, which sits inside a
state. That is why the request has both a `for` (the level you want) and an `in` (the
larger thing it sits inside).

### 1.1 Naming the variables

Census variable codes are precise and unreadable. `B03002_003E` is the estimate of
people who are not Hispanic or Latino and identify as White alone. The `E` on the end
means estimate; the same code ending in `M` gives the margin of error.

We are going to build a **dictionary** that maps each code to a label we can read. A
dictionary pairs a key with a value, written `{key: value}`. We will use it twice: once
to tell the API which variables we want, and once to rename the columns that come back.

Below is the standard breakdown of table B03002, Hispanic or Latino Origin by Race.
Everyone of Hispanic or Latino origin is counted as Hispanic, and every race category is
then non-Hispanic. That is a choice, and it is the most common one in planning practice.
You could code it differently. Be deliberate about it, and say what you did.

In [ ]:
variables_of_interest = {
    'NAME': 'NAME',              # no need to rename this one
    'GEO_ID': 'GEO_ID',          # or this one
    'B03002_001E': 'total',
    'B03002_001M': 'total_moe',
    'B03002_003E': 'nh_white',
    'B03002_003M': 'nh_white_moe',
    'B03002_004E': 'nh_black',
    'B03002_004M': 'nh_black_moe',
    'B03002_005E': 'nh_native',
    'B03002_005M': 'nh_native_moe',
    'B03002_006E': 'nh_asian',
    'B03002_006M': 'nh_asian_moe',
    'B03002_007E': 'nh_pi',
    'B03002_007M': 'nh_pi_moe',
    'B03002_008E': 'nh_1other',
    'B03002_008M': 'nh_1other_moe',
    'B03002_009E': 'nh_multi',
    'B03002_009M': 'nh_multi_moe',
    'B03002_012E': 'hispanic',
    'B03002_012M': 'hispanic_moe',
}

print(f'We are asking for {len(variables_of_interest)} columns.')

Notice that we asked for both the estimate (`E`) and the margin of error (`M`) for every
category. Get in the habit of pulling them together. Next week you will need them.

## 2. Where do these geography codes come from?

Read this section before running anything. It answers the question everybody has at this
point, which is: how would I ever know what to put in the `for` and `in` arguments?

### 2.1 Neighborhoods are not a census geography

This is the important one. The Census Bureau does not publish data for "North Oakland" or
"the Mission" or "West Berkeley," because those are not census geographies. Neighborhoods
are social and political, and their boundaries are contested, which is part of what makes
them interesting.

What the Census Bureau does publish is the **census tract**: a small statistical area,
usually 1,200 to 8,000 people, designed to be reasonably stable over time. So when a
planner says "here is the data for this neighborhood," what they actually did was choose a
set of tracts and add them up. That choice is a judgment call, and a good memo says out
loud which tracts were used and why.

You will make that choice for your own neighborhood after the field trip. Today you are
borrowing one that is already made.

### 2.2 Reading a census geography code

Every tract has an 11-digit identifier, built by nesting:

```
06        001         400500
state     county      tract
California  Alameda   Tract 4005.00
```

The tract piece is six digits because tract numbers can have two decimal places. Tract
4005 is written `400500`. Tract 4005.01 is written `400501`. This trips up everyone at
least once.

The `GEO_ID` column that comes back with every request contains this full code, which is
what you use later to join census data to a map.

### 2.3 Codes you will want today

| Geography | Code | Where it goes |
| --- | --- | --- |
| California | `06` | `state:06` |
| Alameda County | `001` | `county:001` |
| Contra Costa County | `013` | `county:013` |
| San Francisco | `075` | `county:075` |
| City of Oakland | `53000` | `place:53000` |

### 2.4 How to look up your own, when you need to

* **TIGERweb** (https://tigerweb.geo.census.gov/tigerweb/) lets you click anywhere on a
  map and see the tract number for that spot. This is the fastest way to find the tracts
  under a neighborhood you care about.
* **data.census.gov** has a geography picker that will show you tract numbers for a county.
* For state and county codes specifically, this plain-text FIPS list is faster to search
  than anything on the Census site:
  https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt
* Many cities publish their own neighborhood-to-tract crosswalks on their open data
  portals, which saves you the guesswork about boundaries.

You do not need to do any of this today. Bookmark it for after the field trip, when you
pick your own neighborhood.

## 3. Pulling data at three scales

The point of this section is to get the same table three times, at three different
geographic levels, so you can see how the request changes and how the answer changes.

### 3.1 Census tracts (the neighborhood scale)

Five tracts in Alameda County, which together make up a rough version of North Oakland.

In [ ]:
NEIGHBORHOOD_NAME = 'North Oakland'
TRACTS = '400500,400600,400700,400800,400900'
ACS_YEAR = 2024          # 2020 to 2024 5-year estimates

df_tracts = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': f'tract:{TRACTS}', 'in': 'state:06 county:001'},
        year=ACS_YEAR
    )
).rename(columns=variables_of_interest)

df_tracts

Sanity check, every time. Did you get the number of rows you expected? Do the names look
like the places you asked for? Are the totals plausible for a census tract (usually a few
thousand people)?

In [ ]:
print(f'Rows: {len(df_tracts)}')
print(f'Columns: {len(df_tracts.columns)}')
df_tracts[['NAME', 'total']]

In [ ]:
# EXERCISE #3: practice the syntax. Change the county code and pull five tracts from
# somewhere else, using the code table in Section 2.3.
#
# You do not need to know anything about these tracts. This is finger practice, not
# your Assignment 1 neighborhood. Try tract:010101,010102,010202 in county:075
# (San Francisco), or pick your own numbers and see what comes back.
#
# Give your DataFrame a name of your own, not df_tracts.

### 3.2 A city

Cities are "places" in Census vocabulary. Oakland's place code is 53000. Places sit
inside states, so the `in` argument only needs the state.

In [ ]:
df_city = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': 'place:53000', 'in': 'state:06'},
        year=ACS_YEAR
    )
).rename(columns=variables_of_interest)

df_city

### 3.3 A county

Counties sit directly inside states.

In [ ]:
df_county = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': 'county:001', 'in': 'state:06'},
        year=ACS_YEAR
    )
).rename(columns=variables_of_interest)

df_county

### 3.4 Something odd in the county row

Look at the `total_moe` column in the county table you just pulled. It says
**-555555555**.

That is not a margin of error, and it is not a bug. The Census Bureau uses a set of
special values, sometimes called jam values, to flag situations where a normal number
does not apply. `-555555555` means the estimate is **controlled**: the Bureau set this
total to match an official population figure rather than estimating it from the survey
sample, so a margin of error would be meaningless.

You will see others eventually. `-666666666` usually means the estimate could not be
computed, and `-222222222` means a value is not applicable.

The rule for now is simple. **Never do arithmetic on a jam value.** If you divide by one
or average it in, you will get a number, it will be badly wrong, and nothing will warn
you. In Lab 4 we will handle these properly. Today, just notice it and move on.

### 3.5 Before calculating: check the data types

The Census API returns data values as text, even when they look like numbers on the
screen. Python cannot reliably do arithmetic with text, so before we calculate anything we
convert the estimate and margin-of-error columns to numbers.

We leave identifier columns such as `NAME`, `GEO_ID`, `state`, `county`, and `tract` as
text. Their digits identify places; they are not quantities we would add or divide.

In [ ]:
# Check the tract DataFrame before converting anything
df_tracts.info()

In [ ]:
# These are the columns that contain estimates or margins of error.
numeric_cols = [
    col for col in variables_of_interest.values()
    if col not in ['NAME', 'GEO_ID']
]

# Convert those columns in all three DataFrames.
for df in [df_tracts, df_city, df_county]:
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col])

print('Conversion complete.')
df_tracts.info()

### 3.6 One thing to notice, then we move on

Run the cell below. It prints the non-Hispanic Black population estimate and its margin
of error at all three scales. (We use a subgroup rather than the total because, as you saw
above, the county total is controlled and has no usable margin of error.)

You are not calculating anything here. Just look at the two columns together and notice
how large the margin of error is compared to its estimate at the tract level, and how that
relationship changes as the geography gets bigger.

That pattern is the subject of the sampling lectures on September 21 and 23, and of Lab 4.
We are not going to explain it today. Notice it now so that it is familiar when we get
there.

In [ ]:
print('Tract level:')
print(df_tracts[['NAME', 'nh_black', 'nh_black_moe']].to_string(index=False))
print()
print('City level:')
print(df_city[['NAME', 'nh_black', 'nh_black_moe']].to_string(index=False))
print()
print('County level:')
print(df_county[['NAME', 'nh_black', 'nh_black_moe']].to_string(index=False))

## 4. Looking at what you got

Three inspection commands worth building into your habits.

In [ ]:
df_tracts.columns     # what are my columns actually called?

In [ ]:
df_tracts.info()      # what type is each column, and are there missing values?

In [ ]:
df_tracts.describe()  # quick numeric summary

`.columns` is the one that will save you the most time. When you get a `KeyError`, the
reflex is: run `df.columns`, find the real name, copy and paste it.

## 5. Calculating population shares

Raw counts are hard to compare across places of different sizes, so planners usually work
in shares. Pandas will divide two whole columns at once.

In [ ]:
df_tracts['pct_hispanic'] = df_tracts['hispanic'] / df_tracts['total'] * 100

df_tracts[['NAME', 'total', 'hispanic', 'pct_hispanic']]

In [ ]:
# EXERCISE #4: calculate the share for two more groups in df_tracts.

You have just calculated a percentage with no margin of error attached to it. That number
is an estimate built from two other estimates, and it carries uncertainty you cannot see
here. Hold that thought. Next week we put the error bars back on.

## 6. Saving your work

Write your results out so you do not have to re-pull them every time.

In [ ]:
df_tracts.to_csv('lab3_tracts.csv', index=False)
df_city.to_csv('lab3_city.csv', index=False)
df_county.to_csv('lab3_county.csv', index=False)

print('Saved. Check the file browser on the left.')

## 7. If you want more

### 7.1 Every place in California in one call

The `*` wildcard means "all of them."

In [ ]:
df_all_places = pd.DataFrame(
    c.acs5.get(
        list(variables_of_interest.keys()),
        {'for': 'place:*', 'in': 'state:06'},
        year=ACS_YEAR
    )
).rename(columns=variables_of_interest)

print(f'{len(df_all_places)} places in California.')
df_all_places.head()

### 7.2 Finding variables without guessing

You do not have to memorize variable codes. The full list for any year lives at a URL you
can read in your browser:

https://api.census.gov/data/2024/acs/acs5/variables.html

You can also pull the variable list for a single table into pandas.

In [ ]:
var_url = f'https://api.census.gov/data/{ACS_YEAR}/acs/acs5/groups/B03002.json'
var_table = pd.read_json(var_url)
var_table.head(10)

In [ ]:
# EXERCISE #5: pick a table that interests you (median household income is B19013, tenure is
# B25003, means of transportation to work is B08301) and pull it for Oakland, using
# place:53000. Browsing tables now will make your September 22 table list much easier.

### 7.3 A note on syntax you will see elsewhere

Everywhere in this notebook we passed our variables as a **list**, built from the
dictionary:

```python
c.acs5.get(list(variables_of_interest.keys()), ...)
```

That is convenient when you are pulling twenty columns and renaming them all. For a quick
one-off pull you will often see the variables written directly as a **tuple** instead,
in parentheses:

```python
tenure = pd.DataFrame(
    c.acs5.get(
        ('NAME', 'GEO_ID', 'B25003_001E'),
        {'for': 'place:53000', 'in': 'state:06'},
        year=2024
    )
)
tenure
```

Both work. The API does not care which one you hand it. You will meet both forms in
documentation and in other people's code, so it is worth recognizing them.

Watch the quotes in that geography dictionary. A missing opening quote on `'place:53000'`
is one of the easiest typos to make and one of the harder ones to spot.

## Before you leave

* Your key is saved. You will not be asked for it again.
* You can pull an ACS table at three scales.
* You know that neighborhoods are built out of tracts, and roughly where to look tracts up.

**Coming up:** your Python Basics Task Set from Lab 2 and the Lab 3 check-in are both due
**Sunday, September 13**. Your census tracts and ACS table list for Assignment 1 are due
**Tuesday, September 22**, along with the Measuring Urban Change field trip observation
exercise. You will choose your neighborhood after the field trip, so there is nothing to
decide today.

Lab 4 (September 23 and 25) takes the estimates you pulled today and works out what their
margins of error mean, so come with your tracts identified.